# 第8章：传统机器学习模型 ⭐

## 本章学习目标

- 熟练使用 LightGBM、XGBoost 模型
- 掌握模型超参数调优
- 理解特征重要性分析
- 学会模型评估与对比

---

## 8.1 GBDT 模型概述

GBDT (Gradient Boosting Decision Tree) 是量化选股中最常用的模型类型之一。

### 常见 GBDT 实现

| 框架 | 特点 | 适用场景 |
|------|------|----------|
| **LightGBM** | 速度快、内存低 | 大规模数据 |
| XGBoost | 成熟稳定 | 通用场景 |
| CatBoost | 类别特征处理好 | 类别特征多 |

### 为什么选择 LightGBM？

1. **训练速度快**：基于 Histogram 的算法
2. **内存占用低**：特征离散化
3. **精度高**：Leaf-wise 生长策略
4. **支持并行**：特征并行和数据并行

In [ ]:
import qlib
from qlib.data.dataset import DatasetH
from qlib.contrib.data.handler import Alpha158
from qlib.contrib.model.gbdt import LGBModel
from qlib.workflow import R
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 8.2 数据准备

In [ ]:
# 创建数据集
dataset = DatasetH(
    handler={
        "class": "Alpha158",
        "module_path": "qlib.contrib.data.handler",
        "kwargs": {
            "start_time": "2015-01-01",
            "end_time": "2022-12-31",
            "fit_start_time": "2015-01-01",
            "fit_end_time": "2018-12-31",
            "instruments": "csi300",
        },
    },
    segments={
        "train": ("2015-01-01", "2018-12-31"),
        "valid": ("2019-01-01", "2020-06-30"),
        "test": ("2020-07-01", "2022-12-31"),
    },
)

print("数据集创建完成")

# 查看数据集大小
train_data = dataset.prepare("train")
valid_data = dataset.prepare("valid")
test_data = dataset.prepare("test")

print(f"\n训练集: {train_data.shape}")
print(f"验证集: {valid_data.shape}")
print(f"测试集: {test_data.shape}")

In [ ]:
# 查看训练数据结构
train_data.head()

## 8.3 LightGBM 模型配置

### 8.3.1 基础配置

In [ ]:
# 创建 LightGBM 模型
model = LGBModel(
    loss="mse",              # 损失函数
    learning_rate=0.05,      # 学习率
    num_leaves=64,           # 叶子节点数
    max_depth=6,             # 最大深度
    n_estimators=500,        # 树的数量
    colsample_bytree=0.8,    # 特征采样比例
    subsample=0.8,           # 样本采样比例
    random_state=42,
    n_jobs=4,
)

print("模型配置:")
print(f"  learning_rate: {model.learning_rate}")
print(f"  num_leaves: {model.num_leaves}")
print(f"  max_depth: {model.max_depth}")
print(f"  n_estimators: {model.n_estimators}")

### 8.3.2 训练模型

In [ ]:
# 训练模型
print("开始训练 LightGBM 模型...")
model.fit(dataset)
print("训练完成")

### 8.3.3 预测与评估

In [ ]:
# 预测
predictions = model.predict(dataset)

print(f"预测结果形状: {predictions.shape}")
predictions.head()

In [ ]:
# 评估函数
def evaluate_predictions(predictions, labels, group=None):
    """评估预测结果，计算 IC 相关指标"""
    
    pred = np.array(predictions).ravel()
    label = np.array(labels).ravel()
    
    # 移除 NaN
    mask = ~(np.isnan(pred) | np.isnan(label))
    pred = pred[mask]
    label = label[mask]
    
    # IC (Information Coefficient)
    ic = np.corrcoef(pred, label)[0, 1]
    
    # Rank IC
    rank_ic = np.corrcoef(
        np.argsort(np.argsort(pred)), 
        np.argsort(np.argsort(label))
    )[0, 1]
    
    # 计算 ICIR (IC / IC_std)
    # 需要按时间分组计算
    
    return {
        "IC": ic,
        "Rank IC": rank_ic,
        "样本数": len(pred),
    }

# 获取测试集标签
test_data = dataset.prepare("test")
labels = test_data['label']

# 评估
metrics = evaluate_predictions(predictions, labels)

print("模型评估结果:")
print("=" * 40)
for k, v in metrics.items():
    print(f"{k:15s}: {v:.4f}")

## 8.4 特征重要性分析

In [ ]:
# 获取特征重要性
importance = model.get_feature_importance()

# 排序
importance_sorted = importance.sort_values(ascending=False)

print("特征重要性 Top 20:")
print(importance_sorted.head(20))

In [ ]:
# 可视化特征重要性
plt.figure(figsize=(12, 8))

top_n = 30
importance_top = importance_sorted.head(top_n)

plt.barh(range(len(importance_top)), importance_top.values, color='steelblue')
plt.yticks(range(len(importance_top)), importance_top.index)
plt.xlabel('重要性分数')
plt.ylabel('特征')
plt.title(f'LightGBM 特征重要性 Top {top_n}')
plt.tight_layout()
plt.show()

In [ ]:
# 分析不同类型特征的重要性
def categorize_importance(importance):
    """按类别汇总特征重要性"""
    categories = {
        'KBAR': 0,
        'KDJ': 0,
        'RSV': 0,
        'MA': 0,
        'MACD': 0,
        'RSI': 0,
        'PSY': 0,
        'BIAS': 0,
        '其他': 0,
    }
    counts = {k: 0 for k in categories}
    
    for feat, imp in importance.items():
        categorized = False
        for cat in categories:
            if cat != '其他' and cat in feat.upper():
                categories[cat] += imp
                counts[cat] += 1
                categorized = True
                break
        if not categorized:
            categories['其他'] += imp
            counts['其他'] += 1
    
    return pd.DataFrame({
        '总重要性': categories,
        '特征数量': counts,
        '平均重要性': {k: categories[k]/counts[k] if counts[k] > 0 else 0 for k in categories}
    })

cat_importance = categorize_importance(importance)
cat_importance = cat_importance.sort_values('总重要性', ascending=False)

print("按类别汇总的特征重要性:")
cat_importance

In [ ]:
# 可视化类别重要性
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 总重要性
cat_importance['总重要性'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('各类别总特征重要性')
axes[0].set_xlabel('类别')
axes[0].set_ylabel('总重要性')
axes[0].tick_params(axis='x', rotation=45)

# 平均重要性
cat_importance['平均重要性'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('各类别平均特征重要性')
axes[1].set_xlabel('类别')
axes[1].set_ylabel('平均重要性')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 8.5 超参数调优

In [ ]:
# 定义参数搜索空间
param_grid = {
    "learning_rate": [0.01, 0.05, 0.1],
    "num_leaves": [32, 64, 128],
    "max_depth": [4, 6, 8],
    "n_estimators": [100, 300, 500],
}

# 简单的网格搜索
def simple_grid_search(dataset, param_grid, n_combinations=5):
    """简化的网格搜索"""
    import itertools
    from sklearn.model_selection import ParameterGrid
    
    results = []
    
    # 生成参数组合
    param_combinations = list(ParameterGrid(param_grid))[:n_combinations]
    
    for i, params in enumerate(param_combinations):
        print(f"\n训练第 {i+1}/{len(param_combinations)} 组参数: {params}")
        
        # 创建模型
        model = LGBModel(
            loss="mse",
            **params,
            random_state=42,
            n_jobs=4,
        )
        
        # 训练
        model.fit(dataset)
        
        # 预测验证集
        valid_data = dataset.prepare("valid")
        pred = model.predict(dataset)
        
        # 评估
        labels = valid_data['label']
        metrics = evaluate_predictions(pred, labels)
        
        results.append({
            **params,
            **metrics,
        })
    
    return pd.DataFrame(results)

print("开始超参数搜索...")
search_results = simple_grid_search(dataset, param_grid, n_combinations=5)

In [ ]:
# 查看搜索结果
print("超参数搜索结果:")
search_results.sort_values('IC', ascending=False)

In [ ]:
# 选择最优参数
best_params = search_results.loc[search_results['IC'].idxmax()]

print("最优参数:")
print(best_params[['learning_rate', 'num_leaves', 'max_depth', 'n_estimators', 'IC']])

## 8.6 使用最优模型

In [ ]:
# 使用最优参数训练最终模型
best_model = LGBModel(
    loss="mse",
    learning_rate=best_params['learning_rate'],
    num_leaves=int(best_params['num_leaves']),
    max_depth=int(best_params['max_depth']),
    n_estimators=int(best_params['n_estimators']),
    colsample_bytree=0.8,
    subsample=0.8,
    random_state=42,
    n_jobs=4,
)

print("训练最终模型...")
best_model.fit(dataset)
print("训练完成")

In [ ]:
# 使用 Recorder 保存实验
with R.start(experiment_name="lightgbm_best") as recorder:
    
    # 记录参数
    recorder.log_params({
        "model": "LightGBM",
        "features": "Alpha158",
        "learning_rate": best_params['learning_rate'],
        "num_leaves": int(best_params['num_leaves']),
        "max_depth": int(best_params['max_depth']),
        "n_estimators": int(best_params['n_estimators']),
    })
    
    # 预测
    predictions = best_model.predict(dataset)
    
    # 评估
    test_data = dataset.prepare("test")
    labels = test_data['label']
    metrics = evaluate_predictions(predictions, labels)
    
    # 记录指标
    recorder.log_metrics(metrics)
    
    # 保存模型
    recorder.save_object(best_model, name="model.pkl")
    
    print("实验记录完成")
    print(f"\n测试集评估结果:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

## 8.7 IC 时序分析

In [ ]:
# 计算每日 IC
def calculate_daily_ic(predictions, labels):
    """计算每日 IC"""
    # 获取日期索引
    dates = predictions.index.get_level_values('datetime')
    
    daily_ic = []
    for date in dates.unique():
        # 获取当天数据
        pred_day = predictions.xs(date, level='datetime')
        label_day = labels.xs(date, level='datetime')
        
        # 计算相关系数
        if len(pred_day) > 10:  # 至少10个样本
            ic = pred_day.corr(label_day)
            daily_ic.append({
                'date': date,
                'ic': ic,
                'n_samples': len(pred_day),
            })
    
    return pd.DataFrame(daily_ic).set_index('date')

# 计算每日 IC
daily_ic_df = calculate_daily_ic(predictions, labels)

print(f"每日 IC 数量: {len(daily_ic_df)}")
daily_ic_df.head()

In [ ]:
# 可视化每日 IC
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# IC 时序图
axes[0].bar(daily_ic_df.index, daily_ic_df['ic'], 
            color=['green' if x > 0 else 'red' for x in daily_ic_df['ic']],
            alpha=0.7)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].axhline(y=daily_ic_df['ic'].mean(), color='blue', linestyle='--', 
                label=f'平均 IC: {daily_ic_df["ic"].mean():.4f}')
axes[0].set_title('每日 IC 时序图')
axes[0].set_xlabel('日期')
axes[0].set_ylabel('IC')
axes[0].legend()

# IC 累计图
axes[1].plot(daily_ic_df.index, daily_ic_df['ic'].cumsum(), 
             color='blue', linewidth=1.5)
axes[1].set_title('累计 IC')
axes[1].set_xlabel('日期')
axes[1].set_ylabel('累计 IC')

plt.tight_layout()
plt.show()

# IC 统计
print("\nIC 统计:")
print(f"  平均 IC: {daily_ic_df['ic'].mean():.4f}")
print(f"  IC 标准差: {daily_ic_df['ic'].std():.4f}")
print(f"  ICIR: {daily_ic_df['ic'].mean() / daily_ic_df['ic'].std():.4f}")
print(f"  IC > 0 比例: {(daily_ic_df['ic'] > 0).mean():.2%}")

## 8.8 实践练习

In [ ]:
# 练习1: 尝试不同的损失函数
# 对比 'mse' 和 'mae' 的效果

# 你的代码



# 提示：修改 loss 参数

In [ ]:
# 练习2: 实现早停机制
# 当验证集 IC 连续 10 次不提升时停止训练

# 你的代码



# 提示：LGBModel 支持 early_stopping_rounds 参数

In [ ]:
# 练习3: 分析不同时间段的 IC 表现
# 计算 2020 年、2021 年、2022 年的平均 IC

# 你的代码



# 提示：
# ic_2020 = daily_ic_df.loc['2020']['ic'].mean()
# ic_2021 = daily_ic_df.loc['2021']['ic'].mean()
# ic_2022 = daily_ic_df.loc['2022']['ic'].mean()

## 8.9 本章小结

本章我们学习了：

1. **LightGBM 模型使用**：
   - 模型配置与训练
   - 预测与评估

2. **特征重要性分析**：
   - 特征重要性排序
   - 按类别汇总

3. **超参数调优**：
   - 网格搜索
   - 参数选择

4. **IC 分析**：
   - 每日 IC 计算
   - ICIR 指标

### 关键参数说明

| 参数 | 说明 | 典型范围 |
|------|------|----------|
| `learning_rate` | 学习率 | 0.01 - 0.1 |
| `num_leaves` | 叶子节点数 | 31 - 255 |
| `max_depth` | 最大深度 | 4 - 10 |
| `n_estimators` | 树数量 | 100 - 1000 |
| `colsample_bytree` | 特征采样 | 0.6 - 1.0 |
| `subsample` | 样本采样 | 0.6 - 1.0 |

### 下一章预告

下一章我们将学习深度学习模型，包括：
- LSTM 模型
- Transformer 模型
- 模型训练技巧